# 7주차 — Jupyter 종합 데모 (발표용)

> 한국 국고채 10년물 일별 변화량(Δy) 분위수 회귀 예측 — A0 LSTM (Δfeature[t-1])
>
> 본 노트북은 1~6주차 산출물을 발표용으로 통합. 새 계산은 거의 없고 reports/* + figures/* 를 활용.

## 목차

1. 프로젝트 개요 + 차별화 포인트
2. 데이터 + 변수 선정 흐름 (W1-W3)
3. 모델: A0 LSTM 분위수 회귀 (W4-W5)
4. 평가: DM test + 오류 분석 4축 (W6)
5. SHAP + 거시경제 채널 검증
6. 메타-검증 5회 누적 패턴
7. 결론 + 발표 Q&A 대비

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image, Markdown, display
import warnings; warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
ROOT = Path.cwd().parent
REPORT = ROOT/'reports'; FIG = REPORT/'figures'
print('Loaded.')

---

## 1. 프로젝트 개요

### 주제
한국 국고채 10년물 일별 금리 변화량 `Δy_t = (y_t − y_{t-1}) × 100` (bp) 을 거시 변수 8개 입력으로 **LSTM 분위수 회귀** (q=[0.05, 0.5, 0.95]) 로 예측. 1-step ahead causal forecast (입력은 t-1 까지 정보).

### 차별화 포인트

| # | 항목 |
|---|------|
| 1 | **거시경제 도메인** — 학생 프로젝트에서 드문 채권/금리 |
| 2 | **부진 → 진단 → 회복 서사** — W4 raw LSTM 정체 → 입력 비정상성 발견 → A0 (Δfeature[t-1]) 회복 |
| 3 | **DM test 3/3 p<0.0001** (Bonferroni 보정 후) — Naive·XGBoost·LSTM raw 모두 통계적 유의 우월 |
| 4 | **메타-검증 5회 누적** (#30 → #36 → #37 → #40 → #43) — audit 도구 자체의 자가 검증 |
| 5 | **VALIDATION_LOG 43건** = 안내문 최소 3건의 **14.3배** |

---

## 2. 데이터 + 변수 선정 흐름 (W1 → W2 → W3)

In [ ]:
display(Markdown('### W1 — 광역 22개 변수 수집 + EDA + freeze 9개 결정'))
display(Image(FIG/'01_timeseries_by_category.png', width=900))
display(Image(FIG/'02_target_analysis.png', width=900))

In [ ]:
display(Markdown('### W2 — 상관·VIF·Granger 분석 (산출물 3건 사전 마감)'))
display(Image(FIG/'w2_01_correlation_heatmap.png', width=900))
display(Image(FIG/'w2_02_vif.png', width=600))
display(Image(FIG/'w2_03_granger.png', width=900))

In [ ]:
display(Markdown('### W3 — 변수 freeze 8개 확정 (kospi 제외 — Granger p=0.57)'))
display(Markdown('''
| 변수 | 거시경제 채널 |
|------|--------------|
| `kr_treasury_3y` | 한국 단기 금리 (장단기 스프레드) |
| `kr_base_rate` | 한국 통화정책 전이 |
| `us_treasury_10y` | 한미 금리 동조성 |
| `us_fed_funds` | 미국 통화정책 |
| `us_breakeven_10y` | 기대인플레 (BEI) |
| `vix` | 글로벌 위험 신호 |
| `sp500` | 위험자산 흐름 |
| `dxy` | 글로벌 달러 (EM 자본유출 채널) |
'''))

---

## 3. 모델 — A0 LSTM 분위수 회귀 (W4 raw → W5 A0 회복)

In [ ]:
display(Markdown('### 부진 → 진단 → 회복 서사'))
display(Markdown('''
| 단계 | 모델 | test RMSE | Coverage | Dir_Acc |
|------|------|-----------|----------|---------|
| W4 raw LSTM (8 raw vars [t]) | `LSTM(q50)` | 4.535 | 0.824 | 0.495 |
| **W4 진단 (LOG #35)** | features 레벨↔Δy \|r\|<0.05 → 비정상성 | — | — | — |
| **W5 A0 (Δfeat[t-1])** | `A0_Δfeat[t-1]` | **4.195** | **0.902** | **0.638** |
| **개선** | — | **-7.5%** | **+9.5%p** | **+28.9%p** |
'''))
display(Markdown('### 회복의 정량 증거: `Δus_treasury_10y[t-1]` vs `Δy_t` 상관 r=+0.336 (한미 동조성)'))

In [ ]:
# A0 회복 비교 figure
display(Image(FIG/'w4_06_diff_ablation_compare.png', width=950))
display(Markdown('### W5 grid 5×5 (lr × hidden) 탐색 + multi-seed robustness (CV<5%)'))
display(Image(FIG/'w5_01_grid_heatmap.png', width=900))
display(Image(FIG/'w5_02_w5_compare.png', width=950))

---

## 4. 평가 — DM test + 오류 분석 4축

In [ ]:
dm = pd.read_csv(REPORT/'dm_test_w6.csv')
display(Markdown('### DM test (HAC + HLN + Bonferroni α*=0.0167) — **3/3 모두 유의**'))
display(dm[['comparison','rmse_a0','rmse_other','DM_HLN','p_value','bonf']].rename(columns={
    'comparison':'비교','rmse_a0':'RMSE A0','rmse_other':'RMSE 비교','DM_HLN':'DM (HLN)','p_value':'p-value','bonf':'Bonf'
}))
display(Markdown('**해석**: A0 가 Naive·XGBoost·LSTM raw 모두 통계적으로 유의하게 우월 (p<0.0001 × 3). 학부 7주 프로젝트로는 압도적 결과.'))

In [ ]:
display(Markdown('### 오류 분석 4축 (계획서 §6.4)'))
display(Image(FIG/'w6_03_error_analysis_4axis.png', width=950))
display(Markdown('''
- **(a) 방향성 정확도 64.3%** (목표 55%+, +9.3%p 초과)
- **(b) 큰 변동 미예측** \|Δy\|>5 & \|q50\|<1: 59건 (8.8%)
- **(c) Coverage 90.0%** (목표 정확) — 위기 Miss 13.0% vs 정상 7.5% = **1.74× covariate shift 정량**
- **(d) 위기 vs 정상 SHAP 차이**: top-1 = `kr_treasury_3y` (위기 시 +0.0064)
'''))

---

## 5. SHAP + 거시경제 채널 검증

In [ ]:
display(Markdown('### 분위수별 mean\|SHAP\| (top-1 = us_treasury_10y, LOG #35 한미 동조성 재확인)'))
display(Image(FIG/'w6_01_shap_quantile.png', width=950))
display(Image(FIG/'w6_02_shap_time_heatmap.png', width=950))
ch = pd.read_csv(REPORT/'channel_validation_w6.csv')
display(Markdown('### 거시경제 채널 부합 (V6 영역 분리: signed/|SHAP| ratio)'))
display(ch[['feature','hypothesis','actual_sign','signed_abs_ratio','region','match','channel']].rename(columns={
    'feature':'변수','hypothesis':'가설','actual_sign':'실측','signed_abs_ratio':'ratio','region':'영역','match':'부합','channel':'채널'
}))
display(Markdown('''
**해석 (LOG #43)**:
- **strong 1/1**: `kr_base_rate` (signed/\|SHAP\|=86%, 통화정책 전이 ✅)
- **weak 1/2**: `us_fed_funds` 부합 ✅
- **noise 5**: `us_treasury_10y`, `us_breakeven_10y`, `vix`, `sp500`, `dxy` (signed mean 절대값 0.001~0.007 → 결론 보류)

단순 "4/7 부합" 카운트보다 **strong 부합 1개 + noise 5개 결론 보류** 의 정직한 해석이 학술적으로 견고.
'''))

---

## 6. 메타-검증 5회 누적 패턴

In [ ]:
display(Markdown('''
| 라운드 | 발견 | 사용자 직감 |
|--------|------|-------------|
| **#30** (W2) | audit false positive 4건 → 모두 fix | 자체 발견 |
| **#36** (W4 코드 검증) | C1~C4 (CL-05b 정책변수 lag) | "한번더 검증을 해줘" |
| **#37** (#36 메타) | V1~V7 (CL-05c 미국 마감 5변수) | "한번더 검증이 필요할꺼 같은데?" |
| **#40** (W5) | V2 val-overfit + V9 HP 일관성 | "5주차도 검증을 먼저 해야 할 거 같아" |
| **#43** (W6) | V3 lookahead bias + V6 noise 영역 + V7 CL-03 false positive | "검증을 먼저 해줘" |

**패턴**: 매 라운드 사용자 한 줄 직감으로 잔존 결함 발견. **audit 도구는 한 번에 완성되지 않고 누적 강화 필요**.
안내문 §7 "AI 결과 자가 검증" 의 가장 강한 실증.

현재 audit 상태: **CL-01~CL-04, CL-05, CL-06, CL-07 ✅ + CL-05b/c ❌** (W4 raw 잔존 결함 명시화, A0 자동 해결).
'''))

---

## 7. 결론 + 발표 Q&A 대비

In [ ]:
display(Markdown('''
### 결론

1. **A0 LSTM (Δfeature[t-1])** 가 test 구간에서 Naive 대비 **RMSE -9.7%**, Coverage **0.902** 정확 달성, Dir_Acc **0.638** (목표 +15.4%p).
2. **DM test 3/3 p<0.0001** (Bonferroni 보정 후) — 통계적 유의성 강력 입증.
3. **부진 → 진단 → 회복 서사** + **메타-검증 5회 누적** 으로 분석 깊이 차별화.

### 발표 Q&A 대비

**Q1. "Naive 대비 RMSE 0.45 bp 차이는 작지 않나?"**
→ 일별 채권 Δy 는 σ≈4 bp 의 random walk 에 가까움. 같은 절대값에서 (a) DM test p<0.0001 통계적 유의, (b) Coverage 0.902 정확, (c) Dir_Acc 64.3% 가 메인 차별화. 점추정 RMSE 단독 평가는 일별 채권 forecasting 한계 영역.

**Q2. "왜 차분(Δfeature)을 처음부터 안 했는가?"**
→ W4 raw LSTM 부진을 만난 후 사용자 질문 *"결과 부진의 원인 분석은 한 거냐?"* 로 진단 → LOG #35 발견 → A0 회복. 이 흐름 자체가 학습 과정의 정직한 기록 (안내문 §7 직격).

**Q3. "환율 ablation 효과 없는데 왜 안 넣었나?"**
→ A1' 정량 결과 (LOG #39): A0 vs A1' Δ ≈ 0 (모두 std 범위 안). 정성 (정부 개입 §3.4) + 정량 (ablation Δ≈0) 의 이중 근거 (계획서 §3.2(c) 음수 채택 원칙).

**Q4. "위기구간 lookahead 위험은?"**
→ V3 (LOG #43) 에서 발견 → train-only quantile 4.11 bp 로 강제 → 위기 비율 27.5% → 44.5% 정정. 다른 metric 영향 없음.

**Q5. "채널 부합 4/7 인데 만족스럽나?"**
→ V6 (LOG #43) 영역 분리: strong 1/1 (kr_base_rate) + weak 1/2 + noise 5. signed/\|SHAP\|<20% noise 영역은 결론 보류가 정직.

**Q6. "CL-05b/c 잔존 결함은?"**
→ W4 raw LSTM 만 영향 (정책 2 + 미국 마감 5 변수의 raw[t] 사용). 메인 모델 A0 는 `diff().shift(1)` 로 자동 해결. raw LSTM 은 reference baseline 으로 보존.

### 산출물 (GitHub 저장소)

- 노트북 7개 (01~07): EDA → 변수 선정 → 베이스라인 → LSTM → 튜닝 → SHAP+DM → 종합 데모
- 스크립트 8개 (01~08): 데이터 수집·검증 → 누수 audit → 메타-검증
- reports/ 20+ CSV/JSON, 25+ figures
- VALIDATION_LOG 43건, AI_USAGE_LOG 누적
- Streamlit 1페이지 미니 데모: `app/streamlit_app.py`

**GitHub**: https://github.com/donguk77/macro-bond-forecast
'''))